<a href="https://colab.research.google.com/github/yrhutu21/Computer_Vision/blob/main/Batch_Covolution_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
ar = np.array([3,5,6,2,1,5])

In [ ]:
np.convolve(ar, [1,2,3]) # (3,2,1)

array([ 3, 11, 25, 29, 23, 13, 13, 15])


Covolution On the Images from load_digits

In [ ]:
from sklearn.datasets import load_digits

In [ ]:
dt = load_digits(n_class=3)

In [ ]:
dt.keys()

dict_keys(['data', 'target', 'frame', 'feature_names', 'target_names', 'images', 'DESCR'])

In [ ]:
dt.images[0]

array([[ 0.,  0.,  5., 13.,  9.,  1.,  0.,  0.],
       [ 0.,  0., 13., 15., 10., 15.,  5.,  0.],
       [ 0.,  3., 15.,  2.,  0., 11.,  8.,  0.],
       [ 0.,  4., 12.,  0.,  0.,  8.,  8.,  0.],
       [ 0.,  5.,  8.,  0.,  0.,  9.,  8.,  0.],
       [ 0.,  4., 11.,  0.,  1., 12.,  7.,  0.],
       [ 0.,  2., 14.,  5., 10., 12.,  0.,  0.],
       [ 0.,  0.,  6., 13., 10.,  0.,  0.,  0.]])

In [ ]:
# dt.target

In [ ]:
dt.images.shape

(537, 8, 8)

In [ ]:
X = dt.images
Y = dt.target

In [ ]:
X = X.reshape(537,1,8,8)

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam

In [ ]:
X = torch.FloatTensor(X)
Y = torch.LongTensor(Y)

In [ ]:
X.shape

torch.Size([537, 1, 8, 8])

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(64*2*2, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
)

In [ ]:
loss = nn.CrossEntropyLoss()
opt = Adam(model.parameters(), lr=0.001)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
dataset = TensorDataset(X, Y)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
batch_X, _ = next(iter(dataloader))
batch_X.shape

torch.Size([32, 1, 8, 8])

In [ ]:
len(dataloader.dataset)

537

Using Batch Process

In [ ]:
c=0
for epoch in range(100):
    c=c+1
    epoch_loss = 0.0
    for batch_X, batch_Y in dataloader:
        opt.zero_grad()
        yp = model(batch_X)
        ls = loss(yp, batch_Y)
        ls.backward()
        opt.step()
        epoch_loss += ls.item()

    if c%10==0:
      print(epoch_loss)

0.006772944536351133
0.0024733259961067233
0.0010394205683041946
0.0005575061877607368
0.00031870854309090646
0.00020612612252079998
0.00014505186788937863
0.00010455444169110706
7.570491504793608e-05
5.524387540845055e-05


Prediction on new data

In [ ]:
newx = X[112].reshape(1,1,8,8)

In [ ]:
newx.shape

torch.Size([1, 1, 8, 8])

In [ ]:
Yp = model(newx)

In [ ]:
Yp

tensor([[ 0.1824, 13.1003, -4.9616]], grad_fn=<AddmmBackward0>)

In [ ]:
sx = torch.softmax(Yp, dim=1)

In [ ]:
torch.argmax(sx, dim=1)

tensor([1])

Class Based Model


In [ ]:
class MyNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
    self.maxm = nn.MaxPool2d(kernel_size=2, stride=2)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
    self.relu = nn.ReLU()
    self.flatten = nn.Flatten()
    self.linear1 = nn.Linear(64*2*2, 128)
    self.linear2 = nn.Linear(128, 3)

  def forward(self, x):
    x = self.conv1(x)
    x = self.relu(x)
    x = self.maxm(x)
    x = self.conv2(x)
    x = self.relu(x)
    x = self.maxm(x)
    x = self.flatten(x)
    x = self.linear1(x)
    x = self.relu(x)
    x = self.linear2(x)
    return x

In [38]:
class MyNet(nn.Module):
  def __init__(sf):
    super().__init__()
    sf.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)#conv2d :apply 2D convolution operation on image data
    sf.maxm = nn.MaxPool2d(kernel_size=2, stride=2)
    sf.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
    sf.relu = nn.ReLU()
    sf.flatten = nn.Flatten()
    sf.linear1 = nn.Linear(64*2*2, 128)
    sf.linear2 = nn.Linear(128, 3)

  def forward(sf, x):
    print(x.shape)
    x = sf.conv1(x)
    print(x.shape)
    x = sf.relu(x)
    print(x.shape)
    x = sf.maxm(x)
    x = sf.conv2(x)
    x = sf.relu(x)
    x = sf.maxm(x)
    x = sf.flatten(x)
    x = sf.linear1(x)
    x = sf.relu(x)
    x = sf.linear2(x)
    return x

In [ ]:
mod = MyNet()

In [39]:
mod(X)

torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])


tensor([[  9.0621,  -9.3638,  -1.6426],
        [-11.5096,  12.8415,  -1.7288],
        [ -6.5683,  -0.3434,   5.8082],
        ...,
        [ -6.5038,  -3.1261,   8.3559],
        [ -5.7813,  -2.7623,   7.3165],
        [  9.8393,  -9.9256,  -2.2682]], grad_fn=<AddmmBackward0>)

In [ ]:
loss = nn.CrossEntropyLoss()
opt = Adam(mod.parameters(), lr=0.001)

In [ ]:
c=0
for epoch in range(100):
    c=c+1
    opt.zero_grad()
    yp = mod(X)
    ls = loss(yp, Y)
    ls.backward()
    opt.step()
    if c%10==0:
      print(ls.item())

torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
0.22231557965278625
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 32, 8, 8])
torch.Size([537, 1, 8, 8])
torch.Size([537, 3